In [3]:
import torch
from PIL import Image
import matplotlib.pyplot as plt
from timm.data import create_transform, resolve_data_config
import numpy as np

def preprocess_image_for_demonstration(image_path, model):
    """
    Предобрабатывает изображение и показывает результат до и после обработки
    
    Args:
        image_path (str): Путь к изображению
        model: Модель тегирования, на основе которой создаётся трансформация
    
    Returns:
        tuple: (исходное_изображение, обработанное_изображение_tensor)
    """
    # Загружаем изображение
    original_image = Image.open(image_path).convert('RGB')
    
    # Получаем конфигурацию данных для модели
    config = resolve_data_config(model.pretrained_cfg, model=model)
    
    # Создаем трансформацию данных на основе конфигурации модели
    transform = create_transform(**config)
    
    # Применяем трансформацию к изображению
    processed_tensor = transform(original_image)
    
    # Для визуализации конвертируем тензор обратно в изображение
    # Нормализованный тензор нужно денормализовать
    mean = torch.tensor(config.get('mean', [0.485, 0.456, 0.406])).view(3, 1, 1)
    std = torch.tensor(config.get('std', [0.229, 0.224, 0.225])).view(3, 1, 1)
    
    # Денормализуем тензор
    denormalized_tensor = processed_tensor * std + mean
    
    # Клипируем значения к диапазону [0, 1]
    denormalized_tensor = torch.clamp(denormalized_tensor, 0, 1)
    
    # Конвертируем в numpy массив для отображения
    processed_image = denormalized_tensor.permute(1, 2, 0).numpy()
    
    # Отображаем исходное и обработанное изображения
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    
    # Исходное изображение
    axes[0].imshow(np.array(original_image))
    axes[0].set_title('Исходное изображение')
    axes[0].axis('off')
    
    # Обработанное изображение
    axes[1].imshow(processed_image)
    axes[1].set_title(f'Обработанное изображение\n{config.get("input_size", (224, 224))}')
    axes[1].axis('off')
    
    # Информация о трансформации
    resize_size = config.get('input_size', (224, 224))
    interpolation = config.get('interpolation', 'bilinear')
    crop_pct = config.get('crop_pct', 0.875)
    mean_values = config.get('mean', [0.485, 0.456, 0.406])
    std_values = config.get('std', [0.229, 0.224, 0.225])
    
    info_text = (
        f"Применённые трансформации:\n"
        f"- Изменение размера до {resize_size}\n"
        f"- Интерполяция: {interpolation}\n"
        f"- Обрезка: {crop_pct}\n"
        f"- Нормализация: mean={mean_values}, std={std_values}"
    )
    
    plt.figtext(0.5, 0.01, info_text, ha='center', fontsize=10, bbox={"facecolor":"lightgray", "alpha":0.5, "pad":5})
    plt.tight_layout(rect=[0, 0.1, 1, 0.95])
    plt.show()
    
    return original_image, processed_tensor


# Пример использования:
# Подразумевается, что tagger_model уже загружен и доступен
# image_path = "path/to/your/image.jpg"
# original, processed = preprocess_image_for_demonstration(image_path, tagger_model)
# 
# # Дополнительно можно вывести размеры тензора
# print(f"Размер тензора для модели: {processed.shape}")

In [30]:
from Scripts.model import ensure_model_folder, download_model_files, load_model_local_or_remote, load_labels_local_or_remote, load_yolo_model
from pathlib import Path
import torch
from PIL import Image
from timm.data import create_transform, resolve_data_config
import torchvision.transforms as T

# Путь к изображению
path = r"C:/Users/liali/YoloWdTagger/wdv3-timm/Tests/abc/GdjyZ8t_regions/person_detector_2.png"

# Загрузка модели
tagger_model = load_model_local_or_remote("SmilingWolf/wd-eva02-large-tagger-v3", Path("models/taggers/wd-eva02-large-tagger-v3"))

# Получение конфигурации модели
config = resolve_data_config(tagger_model.pretrained_cfg, model=tagger_model)

# Вывод актуальных параметров нормализации
print(f"Актуальные параметры нормализации для модели:")
print(f"Mean: {config.get('mean')}")
print(f"Std: {config.get('std')}")

# Загрузка исходного изображения
original_image = Image.open(path).convert('RGB')

# Создание трансформации точно так же, как в main.py
transform = create_transform(**config)

# Применение трансформации к изображению
processed_tensor = transform(original_image)

# Получаем параметры нормализации из конфигурации
mean = torch.tensor(config.get('mean')).view(3, 1, 1)
std = torch.tensor(config.get('std')).view(3, 1, 1)

# Денормализуем тензор
denormalized_tensor = processed_tensor * std + mean

# Клипируем значения к диапазону [0, 1]
denormalized_tensor = torch.clamp(denormalized_tensor, 0, 1)

# Преобразуем тензор в формат PIL Image
to_pil = T.ToPILImage()
processed_pil_image = to_pil(denormalized_tensor)

# Сохраняем обработанное изображение
output_path = str(Path(path).with_name(f"{Path(path).stem}_processed{Path(path).suffix}"))
processed_pil_image.save(output_path)

print(f"Обработанное изображение сохранено по пути: {output_path}")

Loading model from models\taggers\wd-eva02-large-tagger-v3\wd-eva02-large-tagger-v3\model.safetensors using safetensors
Актуальные параметры нормализации для модели:
Mean: (0.5, 0.5, 0.5)
Std: (0.5, 0.5, 0.5)
Обработанное изображение сохранено по пути: C:\Users\liali\YoloWdTagger\wdv3-timm\Tests\abc\GdjyZ8t_regions\person_detector_2_processed.png


In [12]:
from Scripts.detection import load_detectors_config

detectors = load_detectors_config("detectors.json")

[DetectorConfig(name='person_detector', model_path='person_yolov8s-seg.pt', confidence=0.4, classes=[0], remove_tags_from_full=[], remove_tags_from_region=[], add_tags_to_region={}, exclude_from_region=[], region_gen_threshold=0.45, region_char_threshold=0.8),
 DetectorConfig(name='face_detector', model_path='face_yolov9c.pt', confidence=0.4, classes=[0], remove_tags_from_full=['fang', 'fangs', 'skin fang', 'skin fangs', 'fang out', 'tongue', 'tongue out', 'licking lips', 'aqua eyes', 'black eyes', 'blue eyes', 'brown eyes', 'green eyes', 'grey eyes', 'orange eyes', 'purple eyes', 'pink eyes', 'red eyes', 'white eyes', 'yellow eyes', 'heterochromia', 'multicolored eyes', 'gradient eyes', 'two-tone eyes', 'rainbow eyes', '@ @', 'mismatched irises', 'dashed eyes', 'Pac-man eyes', 'ringed eyes', 'squiggle eyes', 'aqua pupils', 'blue pupils', 'brown pupils', 'green pupils', 'grey pupils', 'orange pupils', 'pink pupils', 'purple pupils', 'red pupils', 'white pupils', 'yellow pupils', 'const